# 01 - Bronze Ingestion

Load the two source CSV files into Bronze Delta tables **without changing the values**.

- Explicit schemas (no `inferSchema`)
- Two metadata columns are added: `_ingested_at` and `_source_file`
- Each run fully refreshes the Bronze tables

## 1. Configuration

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    BooleanType, DoubleType, IntegerType, StringType, StructField, StructType,
)

from common.delta_io import write_delta_table
from configs.etl_config import (
    CROP_FILE_NAME,
    WEATHER_FILE_NAME,
    CROP_CSV_PATH,
    WEATHER_CSV_PATH,
    BRONZE_CROP_TABLE,
    BRONZE_WEATHER_TABLE,
)

## 2. Explicit schemas

In [0]:
crop_schema = StructType([
    StructField("province", StringType(), True),
    StructField("year", IntegerType(), True),
    StructField("quarter", StringType(), True),
    StructField("temperature_2m_mean", DoubleType(), True),
    StructField("temperature_2m_max", DoubleType(), True),
    StructField("temperature_2m_min", DoubleType(), True),
    StructField("precipitation_sum", DoubleType(), True),
    StructField("rain_sum", DoubleType(), True),
    StructField("precipitation_hours", DoubleType(), True),
    StructField("sunshine_duration", DoubleType(), True),
    StructField("shortwave_radiation_sum", DoubleType(), True),
    StructField("reference_evapotranspiration_mm", DoubleType(), True),
    StructField("wind_speed_10m_max", DoubleType(), True),
    StructField("wind_gusts_10m_max", DoubleType(), True),
    StructField("rain_normal", DoubleType(), True),
    StructField("temp_normal", DoubleType(), True),
    StructField("rainfall_deviation_pct", DoubleType(), True),
    StructField("temperature_anomaly_c", DoubleType(), True),
    StructField("oni_index", DoubleType(), True),
    StructField("crop", StringType(), True),
    StructField("production", DoubleType(), True),
    StructField("quarter_num", IntegerType(), True),
    StructField("time_index", IntegerType(), True),
    StructField("production_lag_1", DoubleType(), True),
    StructField("production_lag_2", DoubleType(), True),
    StructField("production_lag_4", DoubleType(), True),
    StructField("temp_anomaly_rolling", DoubleType(), True),
    StructField("rain_anomaly_rolling", DoubleType(), True),
    StructField("enso_phase_la_nina", BooleanType(), True),
    StructField("enso_phase_neutral", BooleanType(), True),
    StructField("crop_group", StringType(), True),
    StructField("growth_rate_yoy", DoubleType(), True),
    StructField("climate_stress", DoubleType(), True),
    StructField("risk_label", StringType(), True),
])

weather_schema = StructType([
    StructField("province", StringType(), True),
    StructField("year", IntegerType(), True),
    StructField("year_quarter", StringType(), True),
    StructField("temperature_2m_mean", DoubleType(), True),
    StructField("temperature_2m_max", DoubleType(), True),
    StructField("temperature_2m_min", DoubleType(), True),
    StructField("precipitation_sum", DoubleType(), True),
    StructField("rain_sum", DoubleType(), True),
    StructField("precipitation_hours", DoubleType(), True),
    StructField("sunshine_duration", DoubleType(), True),
    StructField("shortwave_radiation_sum", DoubleType(), True),
    StructField("reference_evapotranspiration_mm", DoubleType(), True),
    StructField("wind_speed_10m_max", DoubleType(), True),
    StructField("wind_gusts_10m_max", DoubleType(), True),
    StructField("quarter", StringType(), True),
    StructField("rain_normal", DoubleType(), True),
    StructField("temp_normal", DoubleType(), True),
    StructField("rainfall_deviation_pct", DoubleType(), True),
    StructField("temperature_anomaly_c", DoubleType(), True),
    StructField("oni_index", DoubleType(), True),
    StructField("enso_phase", StringType(), True),
])

## 3. Read the CSV files and add metadata columns

In [0]:
def read_source_csv(path: str, schema: StructType, source_file: str):
    """Read one CSV with an explicit schema and tag every row with its source file and load time."""
    return (
        spark.read
        .option("header", True)
        .schema(schema)
        .csv(path)
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.lit(source_file))
    )


crop_raw_df = read_source_csv(CROP_CSV_PATH, crop_schema, CROP_FILE_NAME)
weather_raw_df = read_source_csv(WEATHER_CSV_PATH, weather_schema, WEATHER_FILE_NAME)

crop_raw_df.show(3, truncate=False)
weather_raw_df.show(3, truncate=False)

## 4. Write the Bronze tables

In [0]:
write_delta_table(crop_raw_df, BRONZE_CROP_TABLE)
write_delta_table(weather_raw_df, BRONZE_WEATHER_TABLE)

print(f"Crop rows   : {spark.table(BRONZE_CROP_TABLE).count():,}")
print(f"Weather rows: {spark.table(BRONZE_WEATHER_TABLE).count():,}")

print("Next: 02_silver/01_clean_crop_production")